# P10.6-AI — Notebook 62: split interno subarticular Axial T2

Construye un split 70/15/15 por `study_id`, conserva la asignación del Notebook 58 para los estudios compartidos y sella el `internal_test` para el Notebook 64. No entrena, no abre DICOM y no accede al test oficial. **CPU**.

`humanReviewRequired=true` · `notClinicalDiagnosis=true` · `officialTestAccessed=false`

In [1]:
# 1) Drive, rama y dependencias
from __future__ import annotations
import hashlib, importlib.util, json, os, shutil, subprocess, sys
from datetime import datetime, timezone
from pathlib import Path
req={"numpy":"numpy","pandas":"pandas","sklearn":"scikit-learn"}
missing=[p for m,p in req.items() if importlib.util.find_spec(m) is None]
if missing: subprocess.check_call([sys.executable,"-m","pip","install","--quiet",*missing])
import numpy as np, pandas as pd
from google.colab import drive  # type: ignore
drive.mount("/content/drive",force_remount=False)
URL="https://github.com/EnzoAA004/PFI_MVPTest_Enzo_AImodule.git"; REF="enzo/p10-6-ai-rsna-findings"
ROOT=Path("/content/PFI_MVPTest_Enzo_AImodule")
if not (ROOT/".git").exists(): subprocess.check_call(["git","clone","--branch",REF,"--single-branch",URL,str(ROOT)])
else:
    subprocess.check_call(["git","fetch","origin",REF],cwd=ROOT)
    subprocess.check_call(["git","checkout",REF],cwd=ROOT)
    subprocess.check_call(["git","pull","--ff-only","origin",REF],cwd=ROOT)
SHA=subprocess.check_output(["git","rev-parse","HEAD"],cwd=ROOT,text=True).strip()
sys.path.insert(0,str(ROOT/"ai_service"))
from pfi_ai_service.training.rsna_foraminal_split import run_split, sha256_file
print({"repoRef":REF,"repoSha":SHA,"gpuRequired":False})

Mounted at /content/drive
{'repoRef': 'enzo/p10-6-ai-rsna-findings', 'repoSha': '65cb024c4edca7eaeb69b81030afedde87c8b3c6', 'gpuRequired': False}


In [2]:
# 2) Entradas aprobadas y extensión determinista del split 58
PFI=Path("/content/drive/MyDrive/PFI_MVP"); RES=PFI/"results"/"P10_6_rsna_findings"
N61=RES/"notebook61_subarticular_preflight"; N58=RES/"notebook58_foraminal_split"; OUT=RES/"notebook62_subarticular_split"
M61=N61/"subarticular_candidate_manifest.csv"; S61=N61/"subarticular_preflight_summary.json"
A58=N58/"study_split_assignments.csv"; S58=N58/"split_summary.json"
missing=[str(p) for p in [M61,S61,A58,S58] if not p.is_file()]
if missing: raise FileNotFoundError("Faltan entradas:\n- "+"\n- ".join(missing))
s61=json.loads(S61.read_text()); s58=json.loads(S58.read_text())
g61={"approved":s61.get("approved") is True,"status":s61.get("status")=="APPROVED_FOR_NOTEBOOK_62","next":s61.get("nextNotebook")==62,"official":s61.get("governance",{}).get("officialTestAccessed") is False,"internal":s61.get("governance",{}).get("internalTestAccessed") is False,"notImputed":s61.get("governance",{}).get("missingLabelsImputed") is False}
g58={"approved":s58.get("approved") is True,"next":s58.get("nextNotebook")==59,"sealed":s58.get("governance",{}).get("internalTestSealed") is True,"official":s58.get("governance",{}).get("officialTestAccessed") is False}
if not all(g61.values()) or not all(g58.values()): raise RuntimeError({"notebook61":g61,"notebook58":g58})
if s61.get("outputSha256",{}).get("manifest") not in [None,sha256_file(M61)]: raise RuntimeError("Hash N61 inválido")
if s58.get("outputSha256",{}).get("studyAssignments") not in [None,sha256_file(A58)]: raise RuntimeError("Hash N58 inválido")
manifest=pd.read_csv(M61,dtype={"study_id":str,"coordinate_series_id":str})
required={"study_id","side","level","severity","severity_code","coordinate_series_id","coordinate_instance_number","coordinate_x","coordinate_y","sequence_category"}
if required-set(manifest): raise RuntimeError(f"Faltan columnas: {sorted(required-set(manifest))}")
if manifest.empty or manifest[["study_id","side","level"]].duplicated().any() or manifest.severity.isna().any(): raise RuntimeError("Manifiesto N61 inválido")
if not manifest.sequence_category.eq("axial_t2").all(): raise RuntimeError("Hay filas fuera de Axial T2")
studies=pd.Index(sorted(manifest.study_id.astype(str).unique()),name="study_id"); splits=("train","validation","internal_test"); fractions={"train":.70,"validation":.15,"internal_test":.15}
prior=pd.read_csv(A58,dtype={"study_id":str}); prior["eligible_for_model"]=prior.eligible_for_model.astype(str).str.lower().isin({"true","1","yes"})
prior=prior.loc[prior.eligible_for_model & prior.split.isin(splits),["study_id","split"]].drop_duplicates("study_id").set_index("study_id").split
shared=studies.intersection(prior.index); extra=studies.difference(prior.index)
raw={k:len(studies)*v for k,v in fractions.items()}; target={k:int(np.floor(v)) for k,v in raw.items()}
for k in sorted(splits,key=lambda x:(-(raw[x]-target[x]),splits.index(x)))[:len(studies)-sum(target.values())]: target[k]+=1
counts=prior.loc[shared].value_counts(); deficits={k:target[k]-int(counts.get(k,0)) for k in splits}
if any(v<0 for v in deficits.values()) or sum(deficits.values())!=len(extra): raise RuntimeError({"target":target,"inherited":counts.to_dict(),"extra":len(extra)})
ordered=sorted(extra,key=lambda x:hashlib.sha256(f"2026:{x}".encode()).hexdigest()); added=[]
for split in splits:
    for study_id in ordered[:deficits[split]]: added.append((study_id,split))
    ordered=ordered[deficits[split]:]
complete=pd.concat([prior.loc[shared],pd.Series(dict(added),name="split")]).rename_axis("study_id").sort_index()
if set(complete.index)!=set(studies) or complete.value_counts().to_dict()!=target: raise RuntimeError("Extensión incompleta")
print({"studies":len(studies),"sharedWithNotebook58":len(shared),"newStudies":len(extra),"targetStudyCounts":target,"newAssignments":added})

{'studies': 1974, 'sharedWithNotebook58': 1972, 'newStudies': 2, 'targetStudyCounts': {'train': 1382, 'validation': 296, 'internal_test': 296}, 'newAssignments': [('2492114990', 'train'), ('2780132468', 'train')]}


In [3]:
# 3) Ejecutar auditoría multilabel existente y adaptar la evidencia a Notebook 62
OUT.mkdir(parents=True,exist_ok=True); TMP=Path("/content/pfi_n62_adapter"); shutil.rmtree(TMP,ignore_errors=True); TMP.mkdir()
adapted=manifest.copy(); adapted["usable_for_split"]=True
AM=TMP/"subarticular_manifest_adapter.csv"; AS=TMP/"source_summary_adapter.json"; CS=TMP/"complete_common_split.csv"
adapted.to_csv(AM,index=False); complete.rename("split").reset_index().to_csv(CS,index=False)
AS.write_text(json.dumps({"approved":True,"nextNotebook":58,"governance":{"officialTestAccessed":False},"outputSha256":{"manifest":sha256_file(AM)}}))
base=run_split(AM,AS,CS,OUT,repo_ref=REF,repo_sha=SHA,seed=2026,candidate_count=1)
if base.get("approved") is not True or base.get("splitPolicy")!="reused_notebook54_common_split": raise RuntimeError("El split preservado no superó los gates")
assign=pd.read_csv(OUT/"study_split_assignments.csv",dtype={"study_id":str})
assign["assignment_source"]=np.where(assign.study_id.isin(shared),"reused_notebook58_foraminal_split","deterministic_new_study_assignment")
assign.to_csv(OUT/"study_split_assignments.csv",index=False)
rare=pd.read_csv(OUT/"rare_strata_report.csv"); rare.to_csv(OUT/"split_stratum_support.csv",index=False)
internal=OUT/"internal_test_manifest.csv"
seal={"schemaVersion":"pfi.rsna-subarticular-internal-test-seal.v1","sealedAtUtc":datetime.now(timezone.utc).isoformat(),"manifest":{"path":str(internal),"sha256":sha256_file(internal),"rows":int(len(pd.read_csv(internal))),"studies":int(pd.read_csv(internal,dtype={"study_id":str}).study_id.nunique())},"opened":False,"doNotUseForTraining":True,"authorizedOpenNotebook":64,"sourceNotebook":62,"humanReviewRequired":True,"notClinicalDiagnosis":True,"officialTestAccessed":False}
(OUT/"internal_test_seal.json").write_text(json.dumps(seal,ensure_ascii=False,indent=2,sort_keys=True)+"\n")
base_gates={k:bool(v) for k,v in base["gateResults"].items()}; gates={"sourceNotebook61Approved":all(g61.values()),"sourceManifestHashVerified":True,"sourceNotebook58Approved":all(g58.values()),"source58AssignmentsHashVerified":True,"sharedStudyAssignmentsPreserved":bool((complete.loc[shared]==prior.loc[shared]).all()),"exactTargetStudyCounts":complete.value_counts().to_dict()==target,"allEligibleStudiesAssigned":len(complete)==len(studies),"validSplitNames":set(complete)==set(splits),"supportRulesPassed":base_gates["supportRulesPassed"],"allSeverityClassesInEverySplit":base_gates["allSeverityClassesInEverySplit"],"noStudyLeakage":base_gates["noStudyLeakage"],"rowsConserved":base_gates["rowsConserved"],"noDuplicateStudySideLevelRows":base_gates["noDuplicateStudySideLevelRows"],"internalTestSealed":True,"officialTestNotAccessed":True,"humanReviewRequired":True,"notClinicalDiagnosis":True}
gates={k:bool(v) for k,v in gates.items()}; approved=all(gates.values()); status="APPROVED_FOR_NOTEBOOK_63" if approved else "SUBARTICULAR_SPLIT_REVIEW_REQUIRED"
summary={"schemaVersion":"pfi.rsna-subarticular-split.v1","ticket":"P10.6-AI","notebook":62,"sourceNotebook":61,"createdAtUtc":datetime.now(timezone.utc).isoformat(),"repoRef":REF,"repoSha":SHA,"dataset":"RSNA_LumbarDISC","task":"subarticular_stenosis_left_right","sequence":"Axial T2","status":status,"approved":approved,"nextNotebook":63 if approved else None,"splitPolicy":"preserve_notebook58_shared_studies","seed":2026,"fractions":fractions,"eligibleStudies":len(studies),"eligibleRows":len(manifest),"assignmentExtension":{"targetStudyCounts":target,"sharedStudyCount":len(shared),"newStudyCount":len(extra),"newAssignments":[{"study_id":a,"split":b} for a,b in added]},"splits":base["splits"],"gateResults":gates,"sourceSha256":{"notebook61Manifest":sha256_file(M61),"notebook61Summary":sha256_file(S61),"notebook58Assignments":sha256_file(A58),"notebook58Summary":sha256_file(S58)},"governance":{"commercialUse":False,"humanReviewRequired":True,"notClinicalDiagnosis":True,"autonomousDiagnosis":False,"officialTestAccessed":False,"internalTestAccessed":False,"internalTestSealed":True,"authorizedInternalTestOpenNotebook":64}}
report=["# P10.6-AI — Split subarticular Axial T2","",f"- Estado: `{status}`","- Política: preservar Notebook 58 para estudios compartidos.",f"- Estudios compartidos: {len(shared)}",f"- Estudios nuevos: {len(extra)}",f"- Filas: {len(manifest)}","","## Cohortes",""]+[f"- {k}: {v['studies']} estudios, {v['rows']} filas" for k,v in base["splits"].items()]+["","## Gates",""]+[f"- {k}: `{str(v).lower()}`" for k,v in gates.items()]+["","El internal test queda sellado para el Notebook 64.",""]
(OUT/"split_report.md").write_text("\n".join(report))
outputs={"trainManifest":"train_manifest.csv","validationManifest":"validation_manifest.csv","internalTestManifest":"internal_test_manifest.csv","studyAssignments":"study_split_assignments.csv","splitDistribution":"split_distribution.csv","splitStratumSupport":"split_stratum_support.csv","rareStrata":"rare_strata_report.csv","splitCoverage":"split_coverage.csv","leakageReport":"split_leakage_report.json","internalTestSeal":"internal_test_seal.json","report":"split_report.md"}
summary["outputSha256"]={k:sha256_file(OUT/v) for k,v in outputs.items()}
(OUT/"split_summary.json").write_text(json.dumps(summary,ensure_ascii=False,indent=2,sort_keys=True)+"\n")
print(json.dumps({"status":status,"approved":approved,"nextNotebook":summary["nextNotebook"],"splits":summary["splits"],"assignmentExtension":summary["assignmentExtension"],"gateResults":gates},indent=2,ensure_ascii=False))
if not approved: raise RuntimeError("Revisar split_summary.json antes del Notebook 63")

{
  "status": "APPROVED_FOR_NOTEBOOK_63",
  "approved": true,
  "nextNotebook": 63,
  "splits": {
    "train": {
      "studies": 1382,
      "rows": 13445
    },
    "validation": {
      "studies": 296,
      "rows": 2894
    },
    "internal_test": {
      "studies": 296,
      "rows": 2876
    }
  },
  "assignmentExtension": {
    "targetStudyCounts": {
      "train": 1382,
      "validation": 296,
      "internal_test": 296
    },
    "sharedStudyCount": 1972,
    "newStudyCount": 2,
    "newAssignments": [
      {
        "study_id": "2492114990",
        "split": "train"
      },
      {
        "study_id": "2780132468",
        "split": "train"
      }
    ]
  },
  "gateResults": {
    "sourceNotebook61Approved": true,
    "sourceManifestHashVerified": true,
    "sourceNotebook58Approved": true,
    "source58AssignmentsHashVerified": true,
    "sharedStudyAssignmentsPreserved": true,
    "exactTargetStudyCounts": true,
    "allEligibleStudiesAssigned": true,
    "validSplitName

In [4]:
# 4) Gate final
required=["train_manifest.csv","validation_manifest.csv","internal_test_manifest.csv","study_split_assignments.csv","split_distribution.csv","split_stratum_support.csv","rare_strata_report.csv","split_coverage.csv","split_leakage_report.json","internal_test_seal.json","split_summary.json","split_report.md"]
missing=[x for x in required if not (OUT/x).is_file()]
if missing: raise RuntimeError(f"Faltan outputs: {missing}")
if summary["status"]!="APPROVED_FOR_NOTEBOOK_63" or summary["governance"]["internalTestSealed"] is not True: raise RuntimeError("Gate final no aprobado")
print({"status":summary["status"],"outputs":required,"internalTestSealed":True,"authorizedInternalTestOpenNotebook":64,"officialTestAccessed":False})

{'status': 'APPROVED_FOR_NOTEBOOK_63', 'outputs': ['train_manifest.csv', 'validation_manifest.csv', 'internal_test_manifest.csv', 'study_split_assignments.csv', 'split_distribution.csv', 'split_stratum_support.csv', 'rare_strata_report.csv', 'split_coverage.csv', 'split_leakage_report.json', 'internal_test_seal.json', 'split_summary.json', 'split_report.md'], 'internalTestSealed': True, 'authorizedInternalTestOpenNotebook': 64, 'officialTestAccessed': False}


## Resultado esperado

`APPROVED_FOR_NOTEBOOK_63`. El Notebook 63 solo utilizará train y validation; el internal test se abre una vez en el Notebook 64.